In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/81/7h5x9_fd11n5xw4dczdmcph80000gn/T/ipykernel_31228/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data)
print(data.shape)
data.fillna("",inplace=True)

                                                  text  label
0    DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...      1
1                                             Will do.      0
2    Nora--Cheryl has emailed dozens of memos about...      0
3    Dear Sir=2FMadam=2C I know that this proposal ...      1
4                                                  fyi      0
..                                                 ...    ...
995  So what's the latest? It sounds contradictory ...      0
996  TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...      1
997  Barb I will call to explain. Are you back in t...      0
998    Yang on travelNot free tonite.May work tomorrow      0
999  sbwhoeopSunday February 21 2010 7:42 PMHShaunH...      0

[1000 rows x 2 columns]
(1000, 2)


### Let's divide the training and test set into two partitions

In [4]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data, test_size=0.2, random_state=564)


## Data Preprocessing

In [5]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [6]:
import re

def remove_html(text):
    text = str(text)

    # Remove inline JavaScript / CSS
    text = re.sub(r'<script.*?>.*?</script>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<style.*?>.*?</style>', ' ', text, flags=re.DOTALL | re.IGNORECASE)

    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)

    # Remove remaining HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    return text

data_train["preprocessed_text"] = data_train["text"].apply(remove_html)
data_val["preprocessed_text"] = data_val["text"].apply(remove_html)




- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [7]:
# Your code
def clean_text(text):
    text = str(text)

    # Remove all special characters and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Remove all single characters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)

    # Remove single characters from the start
    text = re.sub(r'^\s*[a-zA-Z]\s+', ' ', text)

    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)

    # Remove prefixed b
    text = re.sub(r'^b\s+', '', text)

    # Convert to lowercase
    text = text.lower()

    return text


data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(clean_text)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(clean_text)



## Now let's work on removing stopwords
Remove the stopwords.

In [8]:
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    text = str(text)
    words = word_tokenize(text)
    words = [word for word in words if word.lower() not in stop_words]
    return " ".join(words)

data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(remove_stopwords)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(remove_stopwords)


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [9]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    text = str(text)
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(lemmatize_text)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(lemmatize_text)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [21]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

ham_text = data_train[data_train["label"] == 0]["preprocessed_text"]
spam_text = data_train[data_train["label"] == 1]["preprocessed_text"]


vectorizer_ham = CountVectorizer()
X_ham_bow = vectorizer_ham.fit_transform(ham_text)

ham_words = pd.DataFrame(
    X_ham_bow.toarray(),
    columns=vectorizer_ham.get_feature_names_out()
)

print(ham_words.sum().sort_values(ascending=False).head(10))

vectorizer_spam = CountVectorizer()
X_spam_bow = vectorizer_spam.fit_transform(spam_text)

spam_words = pd.DataFrame(
    X_spam_bow.toarray(),
    columns=vectorizer_spam.get_feature_names_out()
)

print(spam_words.sum().sort_values(ascending=False).head(10))



pm           117
state        114
would         86
call          78
time          74
secretary     69
president     69
work          65
office        62
one           55
dtype: int64
money          830
account        776
bank           658
fund           634
transaction    472
country        431
business       426
mr             411
nbsp           409
million        392
dtype: int64


## Extra features

In [11]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
print(money_simbol_list)
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

euro|dollar|pound|€|\$


,text,label,preprocessed_text,money_mark,suspicious_words,text_len
129,Sure--I'll call him now.,0,sure call,0,0,9
363,Dear=2C Naturally=2C this letter will come to ...,1,dear naturally letter come surprise since met ...,0,1,2087
378,RELEASE IN PARTB5B6See orig traffic b/I mine,0,release partb see orig traffic mine,0,0,35
883,I hope the bilateral meetings will be across a...,0,hope bilateral meeting across table chair happen,0,0,48
152,MSWesterwelle has a call into you. He would li...,0,mswesterwelle call would like discus recent tr...,0,0,117


## How would work the Bag of Words with Count Vectorizer concept?

Bag of Words with CountVectorizer turn textr into numbers by counting how many times each word appears. The idea is each row representt one document/message and each column is one word from the vocabulary, and eeach value reflects count of that word in that document.

In [12]:
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_bow = vectorizer.transform(data_val["preprocessed_text"])

print(X_train_bow.shape)
print(X_val_bow.shape)


(800, 21155)
(200, 21155)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [13]:

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])


print(X_train_tfidf.shape)
print(X_val_tfidf.shape)


(800, 21155)
(200, 21155)


## And the Train a Classifier?

In [14]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

y_train = data_train["label"]
y_val = data_val["label"]

extra_train = data_train[["money_mark", "suspicious_words", "text_len"]].values
extra_val = data_val[["money_mark", "suspicious_words", "text_len"]].values

# Using TF-IDF
X_train_tfidf_full = hstack([X_train_tfidf, extra_train])
X_val_tfidf_full = hstack([X_val_tfidf, extra_val])

tfidf_model = MultinomialNB()
tfidf_model.fit(X_train_tfidf_full, y_train)

y_pred_tfidf = tfidf_model.predict(X_val_tfidf_full)

print("Accuracy using TF-IDF:", accuracy_score(y_val, y_pred_tfidf))
print(classification_report(y_val, y_pred_tfidf))

# Using Bag of Words
X_train_bow_full = hstack([X_train_bow, extra_train])
X_val_bow_full = hstack([X_val_bow, extra_val])

bow_model = MultinomialNB()
bow_model.fit(X_train_bow_full, y_train)

y_pred_bow = bow_model.predict(X_val_bow_full)

print("Accuracy using BoW:", accuracy_score(y_val, y_pred_bow))
print(classification_report(y_val, y_pred_bow))


Accuracy using TF-IDF: 0.605
              precision    recall  f1-score   support

           0       0.90      0.32      0.47       110
           1       0.53      0.96      0.69        90

    accuracy                           0.60       200
   macro avg       0.72      0.64      0.58       200
weighted avg       0.73      0.60      0.57       200

Accuracy using BoW: 0.835
              precision    recall  f1-score   support

           0       0.94      0.75      0.83       110
           1       0.75      0.94      0.84        90

    accuracy                           0.83       200
   macro avg       0.85      0.84      0.83       200
weighted avg       0.86      0.83      0.83       200



In [15]:

data_full = pd.concat([data_train, data_val], axis=0)

X_full = data_full["preprocessed_text"]
y_full = data_full["label"]

extra_full = data_full[["money_mark", "suspicious_words", "text_len"]].values

tfidf_vectorizer = TfidfVectorizer()
X_full_tfidf = tfidf_vectorizer.fit_transform(X_full)
X_full_tfidf_full = hstack([X_full_tfidf, extra_full])

tfidf_model = MultinomialNB()
tfidf_model.fit(X_full_tfidf_full, y_full)

bow_vectorizer = CountVectorizer()
X_full_bow = bow_vectorizer.fit_transform(X_full)
X_full_bow_full = hstack([X_full_bow, extra_full])

bow_model = MultinomialNB()
bow_model.fit(X_full_bow_full, y_full)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [16]:
# Let's load the testt data
test_data = pd.read_csv("data/kg_test.csv", encoding="latin-1")

In [17]:
# Let'ss appky the same prerocessing step as before on the test data

test_data["preprocessed_text"] = test_data["text"].apply(remove_html)
test_data["preprocessed_text"] = test_data["preprocessed_text"].apply(clean_text)
test_data["preprocessed_text"] = test_data["preprocessed_text"].apply(remove_stopwords)
test_data["preprocessed_text"] = test_data["preprocessed_text"].apply(lemmatize_text)


In [18]:
test_data['money_mark'] = test_data['preprocessed_text'].str.contains(money_simbol_list) * 1
test_data['suspicious_words'] = test_data['preprocessed_text'].str.contains(suspicious_words) * 1
test_data['text_len'] = test_data['preprocessed_text'].apply(lambda x: len(x))


In [19]:
extra_test = test_data[["money_mark", "suspicious_words", "text_len"]].values

# TF-IDF
X_test_tfidf = tfidf_vectorizer.transform(test_data["preprocessed_text"])
X_test_tfidf_full = hstack([X_test_tfidf, extra_test])
y_test_pred_tfidf = tfidf_model.predict(X_test_tfidf_full)

# BoW
X_test_bow = bow_vectorizer.transform(test_data["preprocessed_text"])
X_test_bow_full = hstack([X_test_bow, extra_test])
y_test_pred_bow = bow_model.predict(X_test_bow_full)

print("TF-IDF predictions:")
print(y_test_pred_tfidf[:10])

print("BoW predictions:")
print(y_test_pred_bow[:10])


TF-IDF predictions:
[1 1 1 1 1 1 1 1 1 1]
BoW predictions:
[1 0 0 0 1 1 0 1 1 1]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [20]:
# Your code